# 📊 UAS Data Science — Week 2 Progress
## Data Preprocessing & Feature Engineering

**Dataset:** Sales & Marketing Customer Dataset  
**Tujuan Week 2:** Membersihkan data, menangani missing values, membuat fitur baru, dan menyiapkan data siap modeling

---

## 1. Recap Week 1

Dari hasil EDA di Week 1, ditemukan beberapa masalah yang perlu ditangani:

| Masalah | Kolom | Rencana Penanganan |
|---------|-------|--------------------|
| Missing values | `coupon_code` (40.9%), `age` (8%), `gender` (4.9%), `total_spent` (7%), `satisfaction_score` (4.7%) | Imputasi median/modus, fill 'No Coupon' |
| Anomali usia | `age` < 0 (3 baris), `age` > 100 | Drop baris anomali |
| Class imbalance | `churn`: 84.7% vs 15.3% | SMOTE pada tahap modeling |
| Kolom tanggal | `signup_date`, `last_purchase_date` | Ekstrak fitur numerik baru |
| Kolom kategorik | 7 kolom string | Label/One-Hot Encoding |

---

## 2. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='Set2')

import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')

In [ ]:
df = pd.read_csv('Sales_-_Marketing_customer_dataset.csv')
print(f'Dataset dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom')
df.head(3)

## 3. Data Cleaning

### 3.1 Menghapus Baris dengan Anomali Usia

In [ ]:
print(f'Jumlah baris sebelum cleaning: {len(df):,}')

# Tampilkan baris anomali
anomali_age = df[(df['age'] < 0) | (df['age'] > 100)]
print(f'Baris dengan age tidak wajar  : {len(anomali_age)}')
print(anomali_age[['customer_id', 'age', 'churn']])

# Hapus baris anomali
df = df[~((df['age'] < 0) | (df['age'] > 100))].copy()
df.reset_index(drop=True, inplace=True)

print(f'\nJumlah baris setelah cleaning : {len(df):,}')

### 3.2 Menangani Missing Values

In [ ]:
print('Missing values SEBELUM imputasi:')
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])
print()

In [ ]:
# Imputasi coupon_code: NaN berarti tidak menggunakan kupon
df['coupon_code'] = df['coupon_code'].fillna('No Coupon')

# Imputasi gender dengan modus
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])

# Imputasi age dengan median (robust terhadap outlier)
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)
print(f'Median age untuk imputasi: {median_age}')

# Imputasi total_spent dengan median
median_spent = df['total_spent'].median()
df['total_spent'] = df['total_spent'].fillna(median_spent)
print(f'Median total_spent untuk imputasi: {median_spent:.2f}')

# Imputasi satisfaction_score dengan median
median_sat = df['satisfaction_score'].median()
df['satisfaction_score'] = df['satisfaction_score'].fillna(median_sat)
print(f'Median satisfaction_score untuk imputasi: {median_sat}')

print()
print('Missing values SETELAH imputasi:', df.isnull().sum().sum())
print('✅ Semua missing values berhasil ditangani')

## 4. Feature Engineering

### 4.1 Ekstraksi Fitur dari Kolom Tanggal

In [ ]:
# Parse tanggal
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'])

# Reference date: awal 2025
reference_date = pd.Timestamp('2025-01-01')

# Fitur baru dari tanggal
df['customer_tenure_days'] = (reference_date - df['signup_date']).dt.days
df['recency_days']         = (reference_date - df['last_purchase_date']).dt.days
df['signup_year']          = df['signup_date'].dt.year
df['signup_month']         = df['signup_date'].dt.month

print('Fitur tanggal berhasil dibuat:')
print(df[['customer_id','signup_date','last_purchase_date',
          'customer_tenure_days','recency_days','signup_year']].head(5))

### 4.2 Membuat Fitur Turunan (Derived Features)

In [ ]:
# Pengeluaran per kunjungan
df['spend_per_visit'] = df['total_spent'] / (df['total_visits'] + 1)

# Skor engagement gabungan dari email
df['engagement_score'] = (df['email_open_rate'] * 0.5) + (df['email_click_rate'] * 0.5)

# Rasio refund terhadap total tiket support
df['refund_rate'] = df['refund_requested'] / (df['support_tickets'] + 1)

new_features = ['spend_per_visit', 'engagement_score', 'refund_rate',
                'customer_tenure_days', 'recency_days']

print('Fitur baru yang dibuat:')
print(df[new_features].describe().T[['mean','std','min','max']])
print()
print(f'Total kolom sekarang: {df.shape[1]}')

In [ ]:
# Visualisasi distribusi fitur baru
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat in zip(axes, ['spend_per_visit', 'engagement_score', 'recency_days']):
    for label, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
        ax.hist(df[df['churn'] == label][feat], bins=40, alpha=0.6,
                color=color, density=True,
                label='Tidak Churn' if label == 0 else 'Churn')
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.legend()

plt.suptitle('Distribusi Fitur Baru: Churn vs Tidak Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Encoding Variabel Kategorik

### 5.1 Identifikasi Kolom yang Perlu Di-encode

In [ ]:
# Kolom yang akan di-drop (tidak dipakai untuk modeling)
drop_cols = ['customer_id', 'signup_date', 'last_purchase_date', 'city']
df_model = df.drop(columns=drop_cols).copy()

# Identifikasi kolom kategorik yang tersisa
cat_cols = ['gender', 'country', 'acquisition_channel', 'device_type',
            'subscription_type', 'coupon_code', 'payment_method']

print('Kolom yang akan di-encode (Label Encoding):')
for col in cat_cols:
    print(f'  {col}: {df_model[col].unique().tolist()}')

In [ ]:
# Label Encoding
le = LabelEncoder()
label_encoders = {}

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

print('✅ Label Encoding selesai')
print(f'Shape setelah encoding: {df_model.shape}')
df_model.head(3)

## 6. Feature Scaling

### 6.1 Persiapan Fitur & Target

In [ ]:
# Pisahkan fitur dan target
X = df_model.drop(columns=['churn'])
y = df_model['churn']

print(f'Shape X (fitur): {X.shape}')
print(f'Shape y (target): {y.shape}')
print(f'Distribusi target:')
print(y.value_counts())

In [ ]:
# Kolom yang perlu di-scale (numerik kontinu)
scale_cols = ['age', 'total_visits', 'avg_session_time', 'pages_per_session',
              'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value',
              'delivery_delay_days', 'satisfaction_score', 'nps_score',
              'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq',
              'customer_tenure_days', 'recency_days', 'spend_per_visit',
              'engagement_score', 'refund_rate']

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])

print('✅ StandardScaler diterapkan pada kolom numerik kontinu')
print(f'Contoh nilai setelah scaling (5 kolom pertama):')
print(X_scaled[scale_cols[:5]].describe().T[['mean','std']].round(4))

## 7. Visualisasi Data Bersih

In [ ]:
# Distribusi age sebelum vs sesudah cleaning
df_raw = pd.read_csv('Sales_-_Marketing_customer_dataset.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw['age'].dropna(), bins=40, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribusi Age — SEBELUM Cleaning', fontweight='bold')
axes[0].set_xlabel('Age')

axes[1].hist(df['age'], bins=40, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[1].set_title('Distribusi Age — SETELAH Cleaning', fontweight='bold')
axes[1].set_xlabel('Age')

plt.suptitle('Perbandingan Distribusi Age Sebelum & Sesudah Cleaning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi fitur baru dengan churn
new_feat_corr = df_model[['spend_per_visit','engagement_score','recency_days',
                           'customer_tenure_days','refund_rate','churn']].corr()['churn'].drop('churn')

colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in new_feat_corr.values]

plt.figure(figsize=(9, 4))
bars = plt.barh(new_feat_corr.index, new_feat_corr.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, new_feat_corr.values):
    plt.text(val + (0.001 if val >= 0 else -0.001), bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)
plt.title('Korelasi Fitur Baru dengan Target Churn', fontsize=13, fontweight='bold')
plt.xlabel('Korelasi (Pearson)')
plt.tight_layout()
plt.show()

## 8. Simpan Data yang Sudah Diproses

In [ ]:
# Simpan dataset bersih (sebelum scaling) untuk referensi
df_model.to_csv('data_clean.csv', index=False)

# Simpan dataset siap modeling (sudah di-scale)
X_scaled['churn'] = y.values
X_scaled.to_csv('data_modeling.csv', index=False)

print('✅ File berhasil disimpan:')
print('   - data_clean.csv    : data bersih + fitur baru (belum di-scale)')
print('   - data_modeling.csv : data siap modeling (sudah di-scale)')
print(f'\nShape final: {X_scaled.shape}')
print(f'Total fitur: {X_scaled.shape[1] - 1} (tidak termasuk target)')

## 9. Ringkasan Week 2

### ✅ Yang Telah Dilakukan

| Tahap | Detail |
|-------|--------|
| **Cleaning** | Hapus 3 baris age negatif → data menjadi 14.997 baris |
| **Imputasi** | Median untuk age/total_spent/satisfaction_score; Modus untuk gender; 'No Coupon' untuk coupon_code |
| **Feature Engineering** | 5 fitur baru: `customer_tenure_days`, `recency_days`, `signup_year`, `spend_per_visit`, `engagement_score`, `refund_rate` |
| **Encoding** | Label Encoding pada 7 kolom kategorik |
| **Scaling** | StandardScaler pada 19 kolom numerik kontinu |

### 🗓️ Rencana Week 3
- Train-test split (80:20)
- Penanganan class imbalance dengan **SMOTE**
- Baseline modeling: **Logistic Regression**, **Decision Tree**, **Random Forest**
- Evaluasi: Accuracy, Precision, Recall, F1-Score, ROC-AUC
- Perbandingan performa antar model

---
## 10. Direct Modeling — Tanpa Preprocessing

**Unit Kompetensi:** Membangun model dasar dan menghasilkan prediksi awal

Bagian ini melatih tiga model secara **langsung tanpa preprocessing** (hanya encoding minimal agar model bisa berjalan) untuk melihat performa baseline sebelum dilakukan preprocessing lengkap.

| # | Model | Kategori |
|---|-------|----------|
| 1 | Logistic Regression | Konvensional |
| 2 | Random Forest | Ensemble Bagging |
| 3 | VotingClassifier (LR + SVM + KNN) | Kombinasi Model Konvensional |

> **Catatan:** Data digunakan langsung dari CSV asli — hanya dilakukan encoding kategorik minimal dan drop kolom tanggal/ID yang tidak bisa diproses numerik. Tidak ada imputasi, feature engineering, scaling, atau SMOTE.

In [ ]:
# Import tambahan untuk Direct Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder

print('✅ Library untuk Direct Modeling siap')

### 10.1 Menetapkan Variabel Target (y) dan Fitur Prediktor (X)

In [ ]:
# Load data mentah langsung dari CSV
df_direct = pd.read_csv('Sales_-_Marketing_customer_dataset.csv')

print(f'Dataset asli: {df_direct.shape[0]:,} baris, {df_direct.shape[1]} kolom')
print(f'Missing values: {df_direct.isnull().sum().sum():,}')

# Drop baris missing agar model bisa berjalan (tanpa imputasi)
df_direct = df_direct.dropna()
print(f'Setelah dropna: {df_direct.shape[0]:,} baris')

# Drop kolom yang tidak bisa langsung diproses secara numerik
df_direct = df_direct.drop(columns=['customer_id', 'signup_date', 'last_purchase_date', 'city'])

# Encoding minimal — Label Encoding pada kolom kategorik
cat_cols_direct = df_direct.select_dtypes(include='object').columns.tolist()
print(f'\nKolom kategorik yang di-encode: {cat_cols_direct}')

for col in cat_cols_direct:
    df_direct[col] = LabelEncoder().fit_transform(df_direct[col].astype(str))

# Tetapkan X dan y
X_direct = df_direct.drop(columns=['churn'])   # Semua kolom selain churn
y_direct = df_direct['churn']                  # Target: kolom churn

print(f'\nVariabel target y     : churn')
print(f'Fitur prediktor X     : {X_direct.shape[1]} kolom')
print(f'Distribusi target y   :')
print(y_direct.value_counts())

### 10.2 Train-Test Split

In [ ]:
# Train-test split 80:20 dengan stratify
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_direct, y_direct,
    test_size=0.2,
    random_state=42,
    stratify=y_direct
)

print('=== Hasil Train-Test Split ===')
print(f'Training set : {len(X_train_d):,} baris ({len(X_train_d)/len(X_direct)*100:.1f}%)')
print(f'Testing set  : {len(X_test_d):,} baris ({len(X_test_d)/len(X_direct)*100:.1f}%)')
print()
print('Distribusi churn pada training set:')
print(y_train_d.value_counts())
print()
print('Distribusi churn pada testing set:')
print(y_test_d.value_counts())

### 10.3 Definisi Tiga Model (Tanpa Hyperparameter Tuning)

In [ ]:
# Model 1: Logistic Regression (Konvensional)
model_lr = LogisticRegression(max_iter=1000, random_state=42)

# Model 2: Random Forest (Ensemble Bagging)
model_rf = RandomForestClassifier(random_state=42)

# Model 3: VotingClassifier — LR + SVM + KNN (Kombinasi Model Konvensional)
model_voting = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000, random_state=42)),
        ('svm', SVC(probability=True, random_state=42)),
        ('knn', KNeighborsClassifier())
    ],
    voting='soft'   # soft voting: rata-rata probabilitas dari ketiga model
)

models_direct = {
    'Logistic Regression'  : model_lr,
    'Random Forest'        : model_rf,
    'VotingClassifier\n(LR+SVM+KNN)': model_voting
}

print('Model yang akan dilatih:')
print('  1. Logistic Regression      → Model Konvensional')
print('  2. Random Forest            → Ensemble Bagging')
print('  3. VotingClassifier         → Kombinasi LR + SVM + KNN')
print('\n⚠️  Semua model menggunakan parameter default (tanpa tuning)')

### 10.4 Training & Evaluasi Ketiga Model

In [ ]:
results_direct = {}

for name, model in models_direct.items():
    print(f'Training {name.split(chr(10))[0]}...', end=' ')
    model.fit(X_train_d, y_train_d)
    y_pred = model.predict(X_test_d)

    results_direct[name] = {
        'Accuracy' : accuracy_score(y_test_d, y_pred),
        'Precision': precision_score(y_test_d, y_pred, zero_division=0),
        'Recall'   : recall_score(y_test_d, y_pred, zero_division=0),
        'F1-Score' : f1_score(y_test_d, y_pred, zero_division=0),
        'y_pred'   : y_pred
    }
    print('✅')

print('\n✅ Semua model selesai dilatih')

### 10.5 Tabel Perbandingan Metrik

In [ ]:
# Tabel ringkasan metrik
summary = pd.DataFrame({
    name: {k: v for k, v in res.items() if k != 'y_pred'}
    for name, res in results_direct.items()
}).T
summary.index = ['Logistic Regression', 'Random Forest', 'VotingClassifier (LR+SVM+KNN)']

print('=== Perbandingan Performa — Direct Modeling (Tanpa Preprocessing) ===')
print(summary.to_string())

best = summary['F1-Score'].idxmax()
print(f'\n🏆 Model terbaik (F1-Score): {best} ({summary.loc[best, "F1-Score"]:.4f})')

### 10.6 Visualisasi Perbandingan Metrik

In [ ]:
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_labels  = ['Logistic\nRegression', 'Random\nForest', 'VotingClassifier\n(LR+SVM+KNN)']
colors_models = ['#3b82f6', '#10b981', '#f59e0b']

x     = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))

for i, (name, color) in enumerate(zip(results_direct.keys(), colors_models)):
    vals = [results_direct[name][m] for m in metric_names]
    bars = ax.bar(x + i*width, vals, width, color=color,
                  label=model_labels[i], edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Direct Modeling — Perbandingan Performa Ketiga Model\n(Tanpa Preprocessing, Tanpa Hyperparameter Tuning)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='Baseline 0.5')

plt.tight_layout()
plt.show()

### 10.7 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ['Logistic Regression\n(Konvensional)',
          'Random Forest\n(Ensemble Bagging)',
          'VotingClassifier\n(LR + SVM + KNN)']
cmaps  = ['Blues', 'Greens', 'Oranges']

for ax, (name, res), title, cmap in zip(axes, results_direct.items(), titles, cmaps):
    cm = confusion_matrix(y_test_d, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Tidak Churn', 'Churn'],
                yticklabels=['Tidak Churn', 'Churn'],
                linewidths=0.5, annot_kws={'size': 13})
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')

plt.suptitle('Confusion Matrix — Direct Modeling (Tanpa Preprocessing)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 10.8 Classification Report Detail

In [ ]:
report_titles = [
    'Logistic Regression (Konvensional)',
    'Random Forest (Ensemble Bagging)',
    'VotingClassifier — LR + SVM + KNN'
]

for title, (name, res) in zip(report_titles, results_direct.items()):
    print(f'{"="*55}')
    print(f'  {title}')
    print(f'{"="*55}')
    print(classification_report(
        y_test_d, res['y_pred'],
        target_names=['Tidak Churn', 'Churn']
    ))
    print()

### 10.9 Ringkasan Direct Modeling

| Aspek | Keterangan |
|-------|------------|
| **Data** | Raw CSV — hanya dropna + Label Encoding minimal |
| **Variabel target (y)** | Kolom `churn` (0 = tidak churn, 1 = churn) |
| **Fitur prediktor (X)** | Semua kolom selain churn (setelah drop ID & tanggal) |
| **Split** | 80% train, 20% test (stratified) |
| **Model 1** | Logistic Regression — model konvensional linear |
| **Model 2** | Random Forest — ensemble bagging |
| **Model 3** | VotingClassifier (LR + SVM + KNN) — gabungan konvensional |
| **Tuning** | Tidak ada — semua parameter default |

**Insight:** Recall yang rendah pada Direct Modeling (terutama untuk kelas Churn) menunjukkan pentingnya tahap preprocessing lengkap (imputasi, SMOTE, feature engineering) yang dilakukan pada pipeline utama Week 2–5 untuk meningkatkan kemampuan model mendeteksi pelanggan yang akan churn.